[``DataFrame.query(expr, *, inplace=False, **kwargs)``](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) 函数使用布尔表达式对列进行查询，可以简化我们使用 `loc` 进行查询的写法。

<!-- TEASER_END -->

In [1]:
import pandas as pd
import tushare as ts

pro = ts.pro_api()

In [2]:
data = pro.stock_basic(exchange='SSE', list_status='L', fields='ts_code,symbol,name,area,industry,list_date')
data.head()

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110
1,600004.SH,600004,白云机场,广东,机场,20030428
2,600006.SH,600006,东风汽车,湖北,汽车整车,19990727
3,600007.SH,600007,中国国贸,北京,园区开发,19990312
4,600008.SH,600008,首创环保,北京,环境保护,20000427


如上所示，data 数据集是上海证券交易所的上市公司数据。如果我们要查询股票代码为 600000 的股票，使用 `loc` 方法我们应该利用逻辑运算传入一个布尔表达式：

In [3]:
data.loc[data.symbol == "600000"]

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110


使用 `query`，我们可以这样写：

In [4]:
data.query("symbol == '600000'")

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110


`query` 函数接受一个查询字符串，然后利用 `eval()` 函数对该字符串进行计算，获得一个布尔表达式，再利用布尔表达式实现查询。

`query` 本质与 `loc` 相同，但写法上更清晰、简洁了。同样，`query` 也接受包含复杂逻辑运算的查询字符串，比如筛选地区在上海，同时行业为银行的股票：

In [5]:
data.query("area == '上海' & industry == '银行'")

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110
856,601229.SH,601229,上海银行,上海,银行,20161116
868,601328.SH,601328,交通银行,上海,银行,20070515
946,601825.SH,601825,沪农商行,上海,银行,20210819


`query` 也支持传入变量，使用 `@` 传入：

In [6]:
area = "上海"
industry = "银行"
data.query("area == @area & industry == @industry")

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110
856,601229.SH,601229,上海银行,上海,银行,20161116
868,601328.SH,601328,交通银行,上海,银行,20070515
946,601825.SH,601825,沪农商行,上海,银行,20210819


需要注意的是，对于非有效的 Python 变量名，可以用反引号（``）来使用。比如，如果查询列名为 3D 的列，查询字符串中需要用反引号将 3D 括起来使用。

此外，根据官方文档，DataFrame 实例的 `DataFrame.index` 和 `DataFrame.columns` 属性也被放置在查询命名空间中，因此我们可以把表的索引当作列进行查询。如果索引命名了，可以使用索引的名称在查询中标识它，如果没有命名可以使用 `index`。比如，我们查询 data 数据集中索引为 0 的数据：

In [7]:
data.index

RangeIndex(start=0, stop=2228, step=1)

In [8]:
data.query("index == 0")

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110


列名的命名空间下的方法也可以在查询字符串中使用。比如，我们查询股票名称中包含银行的股票：

In [9]:
data.query("@data.name.str.contains('银行')")

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110
9,600015.SH,600015,华夏银行,北京,银行,20030912
10,600016.SH,600016,民生银行,北京,银行,20001219
28,600036.SH,600036,招商银行,深圳,银行,20020409
723,600908.SH,600908,无锡银行,江苏,银行,20160923
728,600919.SH,600919,江苏银行,江苏,银行,20160802
730,600926.SH,600926,杭州银行,浙江,银行,20161027
732,600928.SH,600928,西安银行,陕西,银行,20190301
786,601009.SH,601009,南京银行,江苏,银行,20070719
825,601128.SH,601128,常熟银行,江苏,银行,20160930
